# Sprint 2 pipeline (chunk → preview → embed → ChromaDB)

Runs **one** cleaned document at a time through chunking, inspection, OpenAI
embedding and storage in the persisted ChromaDB collection.

**The order matters.** Chunking is local and free. Embedding costs money and writes
to the real vector store. So the chunks are produced and previewed *first*, and
nothing is sent to OpenAI until you have looked at Step 2 and are happy with it:

To do another document, change one line in the config cell and run down again.

## Before you run this

1. You need a `.env` at the repository root containing your OpenAI key:

   ```
   OPEN_AI_API_KEY=sk-...
   ```

   `OPENAI_API_KEY` also works — the code accepts either name.

2. **Step 3 only** makes live, paid calls to the OpenAI embeddings API. A typical
   guide is ~100 chunks, which is a single batched request and costs well under a cent.

3. **Step 3 only** writes to the real vector store at `vectorstore/`. Steps 1 and 2
   touch nothing but a `*-CHUNKS.json` file beside the cleaned output.

4. Step 3 is safe to re-run: chunk IDs are deterministic, so a repeat run upserts
   over the same records instead of appending. Running it twice and watching the
   collection count stay put is story 3's live demonstration.

In [1]:
import sys
from pathlib import Path

# Resolve the repo root so `src` imports work no matter where Jupyter started.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.embeddings import chunking_script_v2 as pipeline
from src.embeddings import corpus
from src.vectorstore_client import get_collection

print("repo root:", ROOT)

repo root: /Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI


## Where the corpus is up to

Every cleaned document, how many chunks sit on disk for it, and how many are already
in the collection. Read this to pick the next document to run — anything showing `-`
under *in store* has not been embedded yet.

In [14]:
print(corpus.format_status_table(corpus.corpus_status()))

  document                                                                            on disk   in store
  ------------------------------------------------------------------------------------------------------
  asbestos_assessments                                                                      -          -
  asbestos_removal                                                                          -          -
  concrete_pumping_health_and_safety_guidelines                                             -          -
  conducting_asbestos_surveys                                                               -          -
  excavation_safety                                                                         -          -
  exposure_monitoring_and_health_monitoring                                                 -          -
  health_and_safety_by_design                                                               -          -
  industrial_rope_access_in_new_zealand                

## Config — choose the document

**This is the only cell you edit between runs.** Copy a document name from the table
above into `DOC_DIR`.

In [8]:
# --- the document to run ---
DOC_DIR = ROOT / "data" / "processed" / "managing_work_site_traffic"

# Input: the cleaned elements JSON. Read only — never modified by this notebook.
CLEANED_JSON = next(DOC_DIR.glob("*-CLEANED.json"))

# Output: the chunk records. A derived file alongside the cleaned output, the same
# way *-CLEANED-preview.txt sits next to *-CLEANED.json.
CHUNKS_JSON = CLEANED_JSON.with_name(CLEANED_JSON.name.replace("-CLEANED", "-CHUNKS"))

print("document   :", DOC_DIR.name)
print("cleaned in :", CLEANED_JSON.name)
print("chunks out :", CHUNKS_JSON.name)

document   : managing_work_site_traffic
cleaned in : managing-Work-Site-Traffic-GPG-b0dddbf1-CLEANED.json
chunks out : managing-Work-Site-Traffic-GPG-b0dddbf1-CHUNKS.json


## Step 1 — Chunk

Cleaned elements → chunk records, written to `*-CHUNKS.json` and kept in memory as
`chunks` for the preview below.

**No OpenAI call. No write to the vector store.** Re-run this as often as you like
while tuning chunking; nothing is spent and nothing is persisted to the collection.

The counts underneath are the ones worth glancing at before you pay to embed: chunks
at the size ceiling, chunks under the floor, and chunks that never found a heading.

In [9]:
chunks = pipeline.chunk_document(CLEANED_JSON, CHUNKS_JSON)
SOURCE_FILE = chunks[0]["source_file"]

sizes = sorted(len(c["text"]) for c in chunks)
oversized = [c for c in chunks if len(c["text"]) > pipeline.CHUNK_MAX_CHARS]
undersized = [c for c in chunks if len(c["text"]) < pipeline.CHUNK_MIN_CHARS]
headless = [c for c in chunks if c["section_heading"] == pipeline.FALLBACK_SECTION_HEADING]
tables = [c for c in chunks if c.get("chunk_type") == "table"]

print("\n" + "=" * 62)
print(f"  source file        : {SOURCE_FILE}")
print(f"  chunks created     : {len(chunks)}")
print(f"  chars per chunk    : min {sizes[0]}, median {sizes[len(sizes) // 2]}, max {sizes[-1]}")
print(f"  table chunks       : {len(tables)}")
print("-" * 62)
print(f"  over {pipeline.CHUNK_MAX_CHARS} chars      : {len(oversized)}")
print(f"  under {pipeline.CHUNK_MIN_CHARS} chars      : {len(undersized)}   (one short final chunk is normal)")
print(f"  no section heading : {len(headless)}   (front matter falls back to a placeholder)")
print("=" * 62)

Total records: 555
Chunks created: 60
Saved successfully

  source file        : managing-Work-Site-Traffic-GPG-b0dddbf1.pdf
  chunks created     : 60
  chars per chunk    : min 386, median 1603, max 3369
  table chunks       : 3
--------------------------------------------------------------
  over 4000 chars      : 0
  under 300 chars      : 0   (one short final chunk is normal)
  no section heading : 1   (front matter falls back to a placeholder)


## Step 2 — Preview the first 15 chunks

Read from the in-memory `chunks` list, so this runs **before** anything is embedded.
This is the gate: if the chunks look wrong here, fix the chunking and re-run Step 1
rather than paying to embed them.

The `id` is the deterministic ChromaDB ID — document stem, page, position. It is
computed from the data, so re-running produces exactly the same IDs, which is why
re-ingesting overwrites instead of duplicating.

In [13]:
import textwrap

PREVIEW_COUNT = 15

chunk_ids = pipeline.build_chunk_ids(chunks)

print(f"showing {min(PREVIEW_COUNT, len(chunks))} of {len(chunks)} chunks\n")

for record, chunk_id in list(zip(chunks, chunk_ids))[:PREVIEW_COUNT]:
    print("-" * 78)
    print(f"id      : {chunk_id}")
    pages = (
        f"{record['page_start']}–{record['page_end']}"
        if record.get("page_start") != record.get("page_end")
        else str(record["page_number"])
    )
    print(f"page    : {pages}    type: {record.get('chunk_type', '-')}")
    print(f"section : {record['section_heading']}")
    print(f"chars   : {len(record['text'])}")
    body = record["text"][:4000] + ("..." if len(record["text"]) > 4000 else "")
    print(textwrap.fill(body, width=78, initial_indent="    ", subsequent_indent="    "))

showing 15 of 60 chunks

------------------------------------------------------------------------------
id      : managing-Work-Site-Traffic-GPG-b0dddbf1:p0001:0000
page    : 1–3    type: prose
section : (no section heading)
chars   : 386
    Managing work site traffic GUIDANCE FOR KEEPING HEALTHY AND SAFE AROUND
    VEHICLES AND MOBILE PLANT AT WORK SITES February 2021 New Zealand
    Government WORKSAFE Mahi Haumaru Aotearoa This guide provides practical
    advice on ways to identify and control the health and safety risks
    associated with work site traffic. and thank the stakeholders who have
    contributed Managing work site traffic
------------------------------------------------------------------------------
id      : managing-Work-Site-Traffic-GPG-b0dddbf1:p0001:0001
page    : 1–10    type: prose
section : KEY POINTS
chars   : 1252
    Managing work site traffic GUIDANCE FOR KEEPING HEALTHY AND SAFE AROUND
    VEHICLES AND MOBILE PLANT AT WORK SITES February 2021 New Zealan

## Step 3 — Embed and store

**This is the paid step, and the one that writes to `vectorstore/`.** Only run it once
the preview above looks right.

`stale removed` counts records from an earlier run that this run no longer produces —
for example after changing chunk size. It should be `0` on an unchanged re-run.

**Run this cell twice** to demonstrate idempotency: the first run stores the document,
the second must leave the collection count unchanged.

In [11]:
summary = pipeline.ingest_to_chromadb(CHUNKS_JSON)

# How many records this one document holds, separate from the collection total.
collection = get_collection()
doc_records = len(collection.get(where={"source_file": SOURCE_FILE}, include=[])["ids"])

print("\n" + "=" * 62)
print(f"  collection            : {summary['collection']}")
print(f"  document              : {SOURCE_FILE}")
print(f"  chunks embedded       : {summary['chunks_embedded']}")
print(f"  stale records removed : {summary['stale_removed']}")
print("-" * 62)
print(f"  COUNT BEFORE          : {summary['count_before']}")
print(f"  COUNT AFTER           : {summary['count_after']}")
print("-" * 62)
print(f"  records for this doc  : {doc_records}")
print(f"  pipeline version      : {summary['pipeline_version']}")
print(f"  embedding model       : {summary['embedding_model']}")
print("=" * 62)

if summary["count_before"] == 0:
    print("\nFirst run. Run this cell again — the count must not change.")
elif summary["count_before"] == summary["count_after"]:
    print("\nIDEMPOTENT: the count did not change on this re-run.")
else:
    print("\nCOUNT CHANGED — investigate before signing this story off.")

Loading chunk file...
Chunk validation successful
Embedding 60 chunks with text-embedding-3-small...
  embedding batch 1/1 - 60 texts
Upserting into collection 'hs_construction_v1'...
Stored 60 chunks. Collection count 59 -> 119

  collection            : hs_construction_v1
  document              : managing-Work-Site-Traffic-GPG-b0dddbf1.pdf
  chunks embedded       : 60
  stale records removed : 0
--------------------------------------------------------------
  COUNT BEFORE          : 59
  COUNT AFTER           : 119
--------------------------------------------------------------
  records for this doc  : 60
  pipeline version      : 3.0.1
  embedding model       : text-embedding-3-small

COUNT CHANGED — investigate before signing this story off.


## Done — check the corpus and pick the next document

The document you just ran should now show a count under *in store*. To do the next
one, change `DOC_DIR` in the config cell and run from Step 1 again.

In [12]:
print(corpus.format_status_table(corpus.corpus_status()))

  document                                                                            on disk   in store
  ------------------------------------------------------------------------------------------------------
  asbestos_assessments                                                                      -          -
  asbestos_removal                                                                          -          -
  concrete_pumping_health_and_safety_guidelines                                             -          -
  conducting_asbestos_surveys                                                               -          -
  excavation_safety                                                                         -          -
  exposure_monitoring_and_health_monitoring                                                 -          -
  health_and_safety_by_design                                                               -          -
  industrial_rope_access_in_new_zealand                